In [1]:
import pandas as pd
import numpy as np
import os

In [15]:
# Path to the training csv file
train_file_path = '../data/Industrial Risk and Reliability Assessment/train_data.csv'

In [16]:
# Read the CSV file
df_train = pd.read_csv(train_file_path)

In [17]:
# Check shape and first few rows
print("Train Dataset Shape:", df_train.shape)
display(df_train.head())

Train Dataset Shape: (61249, 27)


,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670,320
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552,319
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213,318
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176,317
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754,316


In [18]:
# Calculate Remaining Useful Life (RUL) for each engine
# Find the maximum cycle for each engine (failure point)
max_cycles = df_train.groupby('engine_id')['cycle'].max().reset_index()
max_cycles.rename(columns={'cycle': 'max_cycle'}, inplace=True)

C:\Users\Bhuvan\AppData\Local\Temp\ipykernel_16484\829838660.py:4: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  max_cycles.rename(columns={'cycle': 'max_cycle'}, inplace=True)


In [19]:
# Merge back with the original training dataframe
df_train = df_train.merge(max_cycles, on='engine_id', how='left')

In [20]:
# Calculate RUL = max_cycle - current cycle
df_train['RUL'] = df_train['max_cycle'] - df_train['cycle']

In [21]:
# Drop the temporary max_cycle column
df_train.drop(columns=['max_cycle'], inplace=True)

C:\Users\Bhuvan\AppData\Local\Temp\ipykernel_16484\3898949634.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_train.drop(columns=['max_cycle'], inplace=True)


In [22]:
# Check the last few columns including RUL for the first few rows
print("Dataset shape after adding RUL:", df_train.shape)
display(df_train[['engine_id', 'cycle', 'RUL']].head())

Dataset shape after adding RUL: (61249, 27)


,engine_id,cycle,RUL
0,1,1,320
1,1,2,319
2,1,3,318
3,1,4,317
4,1,5,316


In [23]:
# Check for any missing/null values in the dataset
missing_values = df_train.isnull().sum().sum()
print("Total Missing Values in Dataset:", missing_values)

Total Missing Values in Dataset: 0


In [24]:
# Check data types and basic info of columns
print("\nDataset Info:")
print(df_train.info())


Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 61249 entries, 0 to 61248
Data columns (total 27 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   engine_id     61249 non-null  int64  
 1   cycle         61249 non-null  int64  
 2   op_setting_1  61249 non-null  float64
 3   op_setting_2  61249 non-null  float64
 4   op_setting_3  61249 non-null  float64
 5   sensor_1      61249 non-null  float64
 6   sensor_2      61249 non-null  float64
 7   sensor_3      61249 non-null  float64
 8   sensor_4      61249 non-null  float64
 9   sensor_5      61249 non-null  float64
 10  sensor_6      61249 non-null  float64
 11  sensor_7      61249 non-null  float64
 12  sensor_8      61249 non-null  float64
 13  sensor_9      61249 non-null  float64
 14  sensor_10     61249 non-null  float64
 15  sensor_11     61249 non-null  float64
 16  sensor_12     61249 non-null  float64
 17  sensor_13     61249 non-null  float64
 18  sensor_14     61249 no

In [25]:
from sklearn.preprocessing import MinMaxScaler

# Define feature columns (excluding engine_id, cycle, and RUL)
feature_cols = [col for col in df_train.columns if col not in ['engine_id', 'cycle', 'RUL']]

In [26]:
# Initialize MinMaxScaler
scaler = MinMaxScaler()

In [27]:
# Fit and transform the feature columns
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])

In [28]:
# Check the scaled data head
print("Data scaled successfully!")
display(df_train[feature_cols].head())

Data scaled successfully!


,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,0.999926,0.997625,1.0,0.000000,0.130347,0.272082,0.212586,0.000000,0.001882,0.002738,...,0.003593,0.993111,0.550773,0.400540,0.0,0.288660,0.627907,1.0,0.015473,0.015881
1,0.476147,0.831591,1.0,0.626985,0.647971,0.634407,0.511781,0.507937,0.498118,0.450787,...,0.450364,0.992395,0.481761,0.351346,0.0,0.608247,0.864693,1.0,0.477968,0.481487
2,0.999900,0.998694,1.0,0.000000,0.123646,0.271245,0.222481,0.000000,0.001255,0.004625,...,0.003202,0.993056,0.531031,0.423787,0.0,0.278351,0.627907,1.0,0.010764,0.018932
3,0.999810,0.997625,1.0,0.000000,0.121351,0.266168,0.224835,0.000000,0.001882,0.004164,...,0.003641,0.993194,0.553707,0.401716,0.0,0.268041,0.627907,1.0,0.012782,0.018724
4,0.595275,0.737173,0.0,0.238089,0.005691,0.033916,0.022025,0.293184,0.208908,0.088924,...,0.087492,0.001405,0.048140,0.920536,0.0,0.030928,0.000000,0.0,0.130172,0.145560


In [29]:
import torch
from torch.utils.data import Dataset, DataLoader

In [34]:
# Create a custom PyTorch Dataset for Turbofan RUL forecasting
class TurbofanDataset(Dataset):
    def __init__(self, dataframe, feature_cols, target_col='RUL', window_size=30):
        self.features = dataframe[feature_cols].values
        self.targets = dataframe[target_col].values
        self.window_size = window_size
        
        # Prepare sliding windows for time-series data
        self.data, self.labels = self._prepare_windows()
        
    def _prepare_windows(self):
        X, y = [], []
        for i in range(len(self.features) - self.window_size):
            X.append(self.features[i:i + self.window_size])
            y.append(self.targets[i + self.window_size])
        return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

In [35]:
# Initialize Dataset with a window size of 30 cycles
window_size = 30
train_dataset = TurbofanDataset(df_train, feature_cols, target_col='RUL', window_size=window_size)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print("Turbofan Dataset & DataLoader created successfully!")
print(f"Total training windows: {len(train_dataset)}")

Turbofan Dataset & DataLoader created successfully!
Total training windows: 61219


C:\Users\Bhuvan\AppData\Local\Temp\ipykernel_16484\2555530782.py:16: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:252.)
  return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).unsqueeze(1)


In [38]:
import torch.nn as nn

# Define the LSTM Model for RUL Prediction
class TurbofanLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, output_dim=1, dropout=0.2):
        super(TurbofanLSTM, self).__init__()
        
        # LSTM layer (corrected hidden_size instead of hidden_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        # Fully connected layers for regression
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, output_dim)
        
    def forward(self, x):
        # Pass through LSTM
        out, _ = self.lstm(x)
        
        # Take the output from the last time step
        out = out[:, -1, :]
        
        # Pass through fully connected layers
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        
        return out

In [39]:
# Initialize the model
input_dim = len(feature_cols)
model = TurbofanLSTM(input_dim=input_dim, hidden_dim=64, num_layers=2, output_dim=1)

print("LSTM Model created successfully!")
print(model)

LSTM Model created successfully!
TurbofanLSTM(
  (lstm): LSTM(24, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=1, bias=True)
)


In [40]:
import torch.optim as optim

# 1. Define Loss Function and Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [41]:
# 2. Training Loop (Running for 3 epochs as a start)
num_epochs = 3
model.train()

print("Training started...")
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        # Zero the gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(batch_X)
        
        # Calculate loss
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()

Training started...


In [42]:
print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_loader):.4f}")

print("Model training completed successfully!")

Epoch [3/3], Loss: 8052.2493
Model training completed successfully!
